# SQL practice 2: NULLs and conditional logic

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-02-nulls-and-conditional-logic.ipynb)

14 exercises on NULL semantics, null-safe comparisons, COALESCE and NULLIF, ANSI-safe functions and CASE. Each exercise names the [reference entries](sql-reference-02-nulls-and-conditional-logic.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 2](sql-reference-02-nulls-and-conditional-logic.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Exercises

8 easy, 6 medium, 0 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Label missing regions](#exercise-1-label-missing-regions) | Easy | [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](sql-reference-02-nulls-and-conditional-logic.ipynb#coalesce-ifnullnvl-nullif-nvl2) |
| 2 | [Three kinds of count](#exercise-2-three-kinds-of-count) | Easy | [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates) |
| 3 | [Customers who never ordered, NULL-safe](#exercise-3-customers-who-never-ordered-null-safe) | Medium | [NULL with `IN` and `NOT IN`](sql-reference-02-nulls-and-conditional-logic.ipynb#null-with-in-and-not-in) |
| 4 | [Everyone outside a department](#exercise-4-everyone-outside-a-department) | Easy | [`IS NULL`, `IS DISTINCT FROM` and `<=>`](sql-reference-02-nulls-and-conditional-logic.ipynb#is-null-is-distinct-from-and-) |
| 5 | [Conversion rate without dividing by zero](#exercise-5-conversion-rate-without-dividing-by-zero) | Medium | [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](sql-reference-02-nulls-and-conditional-logic.ipynb#coalesce-ifnullnvl-nullif-nvl2) |
| 6 | [Bucket orders by size](#exercise-6-bucket-orders-by-size) | Easy | [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case) |
| 7 | [Labels that handle NULL](#exercise-7-labels-that-handle-null) | Easy | [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case) |
| 8 | [Map statuses to stages](#exercise-8-map-statuses-to-stages) | Easy | [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case) |
| 9 | [Average with and without NULLs](#exercise-9-average-with-and-without-nulls) | Medium | [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates) |
| 10 | [Display names with optional parts](#exercise-10-display-names-with-optional-parts) | Easy | [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates) |
| 11 | [Cheapest vendor price](#exercise-11-cheapest-vendor-price) | Medium | [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates), [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case) |
| 12 | [Flags that are not on](#exercise-12-flags-that-are-not-on) | Easy | [Three-valued logic](sql-reference-02-nulls-and-conditional-logic.ipynb#three-valued-logic) |
| 13 | [Parse amounts safely](#exercise-13-parse-amounts-safely) | Medium | [Division by zero and ANSI mode](sql-reference-02-nulls-and-conditional-logic.ipynb#division-by-zero-and-ansi-mode) |
| 14 | [Contact phone by source priority](#exercise-14-contact-phone-by-source-priority) | Medium | [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](sql-reference-02-nulls-and-conditional-logic.ipynb#coalesce-ifnullnvl-nullif-nvl2), [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case) |

## Exercise 1: Label missing regions

**Difficulty:** Easy · about 3 min · **Practises:** [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](sql-reference-02-nulls-and-conditional-logic.ipynb#coalesce-ifnullnvl-nullif-nvl2)

*Reports show a label instead of a blank.*

Return `customer_id`, `name` and `region`, showing `Unknown` when the region is missing, ordered by `customer_id`.

**Expected output**

| customer_id | name | region |
| --- | --- | --- |
| 1 | Acme Corp | North |
| 2 | Bright Retail | North |
| 3 | Cyan Traders | South |
| 4 | Delta Grocers | Unknown |
| 5 | Echo Supplies | West |
| 6 | Foxtrot Foods | West |
| 7 | Gamma Sports | East |
| 8 | Hotel Homeware | East |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, name, coalesce(region, 'Unknown') AS region
    FROM customers
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Three kinds of count

**Difficulty:** Easy · about 3 min · **Practises:** [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates)

*Interviewers love asking what `count(*)`, `count(col)` and `count(DISTINCT col)` each return.*

Return one row with `employees` (all rows), `with_dept` (employees that have a department) and `departments` (distinct departments).

**Expected output**

| employees | with_dept | departments |
| --- | --- | --- |
| 10 | 9 | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT count(*) AS employees, count(dept) AS with_dept, count(DISTINCT dept) AS departments
    FROM employees
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: Customers who never ordered, NULL-safe

**Difficulty:** Medium · about 10 min · **Practises:** [NULL with `IN` and `NOT IN`](sql-reference-02-nulls-and-conditional-logic.ipynb#null-with-in-and-not-in)

*Orphaned records with a NULL foreign key break `NOT IN` silently.*

Return the `customer_id` and `name` of customers with no orders, ordered by `customer_id`. The query must still be correct if `orders` contains rows with a NULL `customer_id`.

**Expected output**

| customer_id | name |
| --- | --- |
| 7 | Gamma Sports |
| 8 | Hotel Homeware |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Everyone outside a department

**Difficulty:** Easy · about 5 min · **Practises:** [`IS NULL`, `IS DISTINCT FROM` and `<=>`](sql-reference-02-nulls-and-conditional-logic.ipynb#is-null-is-distinct-from-and-)

*"Not in Engineering" must include people with no department.*

Return `name` and `dept` of employees who are not in Engineering, including those without a department, ordered by `name`.

**Expected output**

| name | dept |
| --- | --- |
| Eli | Sales |
| Farah | Sales |
| Gus | Sales |
| Hana | Finance |
| Ivan | Finance |
| Jo | null |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name, dept
    FROM employees
    WHERE dept IS DISTINCT FROM 'Engineering'
    ORDER BY name
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: Conversion rate without dividing by zero

**Difficulty:** Medium · about 10 min · **Practises:** [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](sql-reference-02-nulls-and-conditional-logic.ipynb#coalesce-ifnullnvl-nullif-nvl2)

*Funnel metrics divide by counts that can be zero.*

Per `user_id` in `events`, return `views`, `purchases` and `conversion` = purchases / views rounded to 2 decimals, NULL when the user has no views. Count every event row (including the retry). Order by `user_id`.

**Expected output**

| user_id | views | purchases | conversion |
| --- | --- | --- | --- |
| 1 | 1 | 1 | 1.0 |
| 2 | 1 | 1 | 1.0 |
| 3 | 2 | 0 | 0.0 |
| 4 | 0 | 1 | null |
| 5 | 3 | 1 | 0.33 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id,
           count_if(event = 'view')     AS views,
           count_if(event = 'purchase') AS purchases,
           round(count_if(event = 'purchase') / nullif(count_if(event = 'view'), 0), 2) AS conversion
    FROM events
    GROUP BY user_id
    ORDER BY user_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Bucket orders by size

**Difficulty:** Easy · about 5 min · **Practises:** [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case)

*Bands make distributions readable.*

Return `order_id`, `amount` and `size`: `small` under 100, `medium` from 100 to under 500, `large` from 500. Order by `order_id`.

**Expected output**

| order_id | amount | size |
| --- | --- | --- |
| 1 | 120.00 | medium |
| 2 | 84.50 | small |
| 3 | 30.00 | small |
| 4 | 260.00 | medium |
| 5 | 55.25 | small |
| 6 | 149.00 | medium |
| 7 | 310.00 | medium |
| 8 | 89.90 | small |
| 9 | 1180.00 | large |
| 10 | 74.00 | small |
| 11 | 210.00 | medium |
| 12 | 96.40 | small |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, amount,
           CASE WHEN amount >= 500 THEN 'large'
                WHEN amount >= 100 THEN 'medium'
                ELSE 'small' END AS size
    FROM orders
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Labels that handle NULL

**Difficulty:** Easy · about 5 min · **Practises:** [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case)

*A simple CASE cannot match NULL.*

Return `name`, `team` (the department, or `Unassigned` when missing) and `tenure`: `veteran` for hires before 2020, otherwise `recent`. Order by `emp_id`.

**Expected output**

| name | team | tenure |
| --- | --- | --- |
| Ada | Engineering | veteran |
| Brett | Engineering | veteran |
| Chen | Engineering | recent |
| Dana | Engineering | recent |
| Eli | Sales | veteran |
| Farah | Sales | recent |
| Gus | Sales | recent |
| Hana | Finance | veteran |
| Ivan | Finance | recent |
| Jo | Unassigned | recent |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT name,
           CASE WHEN dept IS NULL THEN 'Unassigned' ELSE dept END AS team,
           CASE WHEN hire_date < DATE'2020-01-01' THEN 'veteran' ELSE 'recent' END AS tenure
    FROM employees
    ORDER BY emp_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Map statuses to stages

**Difficulty:** Easy · about 3 min · **Practises:** [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case)

*Workflow states map to ordered stages; unknown states must stand out.*

Return `order_id`, `status` and `stage` (placed 1, paid 2, shipped 3, anything else -1), ordered by `stage` then `order_id`.

**Expected output**

| order_id | status | stage |
| --- | --- | --- |
| 3 | returned | -1 |
| 8 | returned | -1 |
| 5 | placed | 1 |
| 11 | placed | 1 |
| 1 | paid | 2 |
| 4 | paid | 2 |
| 7 | paid | 2 |
| 10 | paid | 2 |
| 2 | shipped | 3 |
| 6 | shipped | 3 |
| 9 | shipped | 3 |
| 12 | shipped | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, status,
           CASE status WHEN 'placed' THEN 1 WHEN 'paid' THEN 2 WHEN 'shipped' THEN 3 ELSE -1 END AS stage
    FROM orders
    ORDER BY stage, order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Average with and without NULLs

**Difficulty:** Medium · about 5 min · **Practises:** [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates)

*Treating missing as zero is a business decision, and it changes the number.*

Using the `scores` view below, return one row with `scored` (non-NULL scores), `students`, `avg_known` (average of known scores) and `avg_missing_as_zero`, both rounded to 2.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW scores AS SELECT * FROM VALUES
  ('ana', 90), ('ben', NULL), ('cy', 70), ('dee', NULL), ('eve', 80)
AS scores(student, score)
""")
show("SELECT * FROM scores")

**Expected output**

| scored | students | avg_known | avg_missing_as_zero |
| --- | --- | --- | --- |
| 3 | 5 | 80.0 | 48.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT count(score) AS scored, count(*) AS students,
           round(avg(score), 2) AS avg_known,
           round(avg(coalesce(score, 0)), 2) AS avg_missing_as_zero
    FROM scores
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Display names with optional parts

**Difficulty:** Easy · about 5 min · **Practises:** [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates)

*Concatenating a NULL wipes out the whole string.*

Return `customer_id` and `label`: `name (region)`, or just the name when the region is missing. Order by `customer_id`, customers 1 to 5 only.

**Expected output**

| customer_id | label |
| --- | --- |
| 1 | Acme Corp (North) |
| 2 | Bright Retail (North) |
| 3 | Cyan Traders (South) |
| 4 | Delta Grocers |
| 5 | Echo Supplies (West) |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id,
           CASE WHEN region IS NULL THEN name ELSE concat(name, ' (', region, ')') END AS label
    FROM customers
    WHERE customer_id <= 5
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Cheapest vendor price

**Difficulty:** Medium · about 10 min · **Practises:** [NULLs in arithmetic, strings and aggregates](sql-reference-02-nulls-and-conditional-logic.ipynb#nulls-in-arithmetic-strings-and-aggregates), [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case)

*Comparing columns where some vendors have no quote.*

Using `quotes` below, return `sku`, `best_price` (lowest non-NULL price) and `vendor` (`a`, `b` or `c`; on a tie prefer a, then b), or NULL for both when no vendor quoted. Order by `sku`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW quotes AS SELECT * FROM VALUES
  ('SKU-1', 10.0, 12.0, 9.5), ('SKU-2', NULL, 7.0, 7.0), ('SKU-3', NULL, NULL, NULL), ('SKU-4', 5.0, NULL, 6.0)
AS quotes(sku, vendor_a, vendor_b, vendor_c)
""")
show("SELECT * FROM quotes")

**Expected output**

| sku | best_price | vendor |
| --- | --- | --- |
| SKU-1 | 9.5 | c |
| SKU-2 | 7.0 | b |
| SKU-3 | null | null |
| SKU-4 | 5.0 | a |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT sku,
           least(vendor_a, vendor_b, vendor_c) AS best_price,
           CASE least(vendor_a, vendor_b, vendor_c)
                WHEN vendor_a THEN 'a' WHEN vendor_b THEN 'b' WHEN vendor_c THEN 'c' END AS vendor
    FROM quotes
    ORDER BY sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Flags that are not on

**Difficulty:** Easy · about 3 min · **Practises:** [Three-valued logic](sql-reference-02-nulls-and-conditional-logic.ipynb#three-valued-logic)

*Feature flags with NULL mean "never set", which is not "enabled".*

Using `flags` below, return the `user_id`s whose `enabled` flag is not TRUE (FALSE or NULL), ordered.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW flags AS SELECT * FROM VALUES (1, TRUE), (2, FALSE), (3, NULL), (4, TRUE) AS flags(user_id, enabled)")

**Expected output**

| user_id |
| --- |
| 2 |
| 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT user_id
    FROM flags
    WHERE enabled IS NOT TRUE
    ORDER BY user_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Parse amounts safely

**Difficulty:** Medium · about 5 min · **Practises:** [Division by zero and ANSI mode](sql-reference-02-nulls-and-conditional-logic.ipynb#division-by-zero-and-ansi-mode)

*Raw feeds contain values that do not parse; ANSI mode would fail the whole job.*

Using `raw_payments` below, return `payment_id`, `raw` and `amount` (the value as `DECIMAL(10,2)`, NULL when it does not parse), ordered by `payment_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW raw_payments AS SELECT * FROM VALUES
  (1, '12.50'), (2, '7'), (3, 'n/a'), (4, NULL), (5, '1,200.00'), (6, '-3.1')
AS raw_payments(payment_id, raw)
""")

**Expected output**

| payment_id | raw | amount |
| --- | --- | --- |
| 1 | 12.50 | 12.50 |
| 2 | 7 | 7.00 |
| 3 | n/a | null |
| 4 | null | null |
| 5 | 1,200.00 | null |
| 6 | -3.1 | -3.10 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT payment_id, raw, try_cast(trim(raw) AS DECIMAL(10,2)) AS amount
    FROM raw_payments
    ORDER BY payment_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Contact phone by source priority

**Difficulty:** Medium · about 10 min · **Practises:** [`COALESCE`, `IFNULL`/`NVL`, `NULLIF`, `NVL2`](sql-reference-02-nulls-and-conditional-logic.ipynb#coalesce-ifnullnvl-nullif-nvl2), [`CASE`](sql-reference-02-nulls-and-conditional-logic.ipynb#case)

*Golden-record logic picks the best available value from several systems.*

Using `contacts` below, return `customer_id`, `phone` (CRM first, then billing, then support; blank strings count as missing) and `source` (`crm`, `billing`, `support` or `none`). Order by `customer_id`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS SELECT * FROM VALUES
  (1, '555-0101', '555-0199', NULL), (2, '  ', '555-0202', '555-0299'), (3, NULL, NULL, '555-0303'), (4, '', NULL, NULL)
AS contacts(customer_id, crm_phone, billing_phone, support_phone)
""")

**Expected output**

| customer_id | phone | source |
| --- | --- | --- |
| 1 | 555-0101 | crm |
| 2 | 555-0202 | billing |
| 3 | 555-0303 | support |
| 4 | null | none |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id,
           coalesce(nullif(trim(crm_phone), ''), nullif(trim(billing_phone), ''), nullif(trim(support_phone), '')) AS phone,
           CASE WHEN nullif(trim(crm_phone), '') IS NOT NULL THEN 'crm'
                WHEN nullif(trim(billing_phone), '') IS NOT NULL THEN 'billing'
                WHEN nullif(trim(support_phone), '') IS NOT NULL THEN 'support'
                ELSE 'none' END AS source
    FROM contacts
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 2: NULLs and conditional logic](sql-reference-02-nulls-and-conditional-logic.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).